In [ ]:
import os

# Set working directory and GPU before importing JAX
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import jax
import jax.numpy as jnp
from jax import vmap

from jaxpi.models import (
    create_lr_schedule,
    create_optimizer,
    create_arch,
    create_train_state,
)
from jaxpi.checkpointing import (
    create_checkpoint_manager,
    save_checkpoint,
    restore_checkpoint,
)

import models
from utils import get_dataset, inflow_profile

jax.config.update("jax_default_matmul_precision", "highest")

In [ ]:
# Load config
from configs import baseline, fixed_pseudo_time, pseudo_time
config = baseline.get_config()

In [ ]:
# Get dataset
(
    u_ref,
    v_ref,
    p_ref,
    coords,
    inflow_coords,
    outflow_coords,
    wall_coords,
    nu,
) = get_dataset()

u_inflow, _ = inflow_profile(inflow_coords[:, 1])

In [ ]:
# Initialize the model
lr = create_lr_schedule(config.optim)
tx = create_optimizer(config.optim, lr)
arch = create_arch(config.arch)
state = create_train_state(config, tx, arch)

# Initialize model
model = models.NavierStokes2D(config, lr, tx, arch, state, u_inflow, inflow_coords, outflow_coords, wall_coords, nu)

In [ ]:
# Create checkpoint manager for the first time window
ckpt_path = os.path.join(os.getcwd(), config.wandb.name, "ckpt")
ckpt_mngr = create_checkpoint_manager(config.saving, ckpt_path)

# restore checkpoint
state = restore_checkpoint(ckpt_mngr, state)
print("Restore checkpoint from step: ", int(state.step))

In [ ]:
# Evaluate on test set
u_pred, v_pred, _ = vmap(model.neural_net, (None, 0, 0))(state.params, coords[:, 0], coords[:, 1])

u_error = jnp.linalg.norm(u_pred - u_ref) / jnp.linalg.norm(u_ref)
v_error = jnp.linalg.norm(v_pred - v_ref) / jnp.linalg.norm(v_ref)

print("Relative L2 error for u: ", u_error)
print("Relative L2 error for v: ", v_error)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as tri

x = coords[:, 0]
y = coords[:, 1]
triang = tri.Triangulation(x, y)

U_pred = jnp.sqrt(u_pred**2 + v_pred**2)
U_ref = jnp.sqrt(u_ref**2 + v_ref**2)

fig1 = plt.figure(figsize=(16, 6))
plt.subplot(3, 1, 1)
plt.tricontourf(triang, U_ref, cmap="jet", levels=100)
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Exact")
plt.tight_layout()

plt.subplot(3, 1, 2)
plt.tricontourf(triang, U_pred, cmap="jet", levels=100)
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Predicted u(x, y)")
plt.tight_layout()

plt.subplot(3, 1, 3)
plt.tricontourf(triang, jnp.abs(U_ref - U_pred), cmap="jet", levels=100)
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Absolute error")
plt.tight_layout()